# 06.10 — NaN-masked reconstruction

Some recording channels are removed during preprocessing. Rather than delete them (which would ragged the array shapes), they're stored as **`NaN`** — a channel-shaped hole. That single design choice forces a **two-layered NaN strategy** (Critical Note #38): the encoder must never *see* a `NaN` (it would propagate through every op and poison the whole network), but the reconstruction loss must *know where the NaNs are* so it doesn't penalize the decoder for missing channels. This notebook is that two-tensor contract — and the empirical-parity story that goes with it.

This notebook covers:

* The two layers: `NaNToZero` at the input, NaN-masking in the loss.
* The two-tensor contract: NaN-zeroed input, NaN-preserving target.
* Why `torch.where`, not `mask * diff` (the `NaN·0 = NaN` trap).
* The normalization the migration plan got *wrong* — and how an empirical probe caught it.

**Prerequisites:** [02.8 NaN handling](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb), [06.2 VAE + ELBO](06.2_vae_and_the_elbo.ipynb).

## Section 1 — What MATLAB does

Removed channels are `NaN` on disk. MATLAB handles them in two places:

```matlab
% (a) Input path — cgg_setNaNToValue(x, 0): zero the NaNs BEFORE the encoder
X_encoder = cgg_setNaNToValue(X, 0);        % encoder never sees NaN

% (b) Reconstruction loss — mask the NaNs of the ORIGINAL target
loss = 0.5 * l2loss(Y, T, Mask = ~isnan(T));  % removed channels contribute 0
```

Two different tensors: the encoder gets the *NaN-zeroed* input; the loss gets the *NaN-preserving* target. Drop either and training silently breaks — a `NaN` reaches the encoder and poisons everything, or the loss penalizes the decoder for channels that were never there. The `l2loss` normalization hides a subtlety that was the single highest-risk ambiguity in the whole port (§2.4).

## Section 2 — The Python concepts you need

### 2.1 — Layer (a): `NaNToZero` at the input

A `NaN` is contagious: any arithmetic touching it yields `NaN` ([02.8 §2.1](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb)). If a `NaN` channel entered the encoder, the first `matmul` would spread it across *every* output feature, and the whole forward pass would be `NaN`. So layer (a) replaces `NaN` with `0` *before* the encoder — a removed channel becomes a silent zero-signal channel, which the network handles fine.

In [ ]:
import torch
from neural_data_decoding.models.layers.nan_to_zero import NaNToZero

x = torch.tensor([[1.0, float("nan"), 3.0],
                  [float("nan"), 5.0, 6.0]])     # two removed-channel holes
zeroed = NaNToZero()(x)
print("input (with NaN holes):\n", x)
print("after NaNToZero (encoder-safe):\n", zeroed)
print("any NaN left?", bool(torch.isnan(zeroed).any()), "→ encoder can run.")

### 2.2 — Layer (b): NaN-masking in the loss

The reconstruction loss gets the *original* target — `NaN`s intact. It derives a mask `~isnan(target)` and only scores positions that were real channels. The removed channels contribute zero to the loss, so the decoder is never penalized for failing to reconstruct data that isn't there.

### 2.3 — The `NaN·0 = NaN` trap — why `torch.where`, not `mask * diff`

The obvious way to "zero out" masked positions is `mask * diff`. It's a bug: in IEEE arithmetic, `NaN · 0 = NaN`, not `0`. Multiplying a masked `NaN` difference by `0` leaves the `NaN`, which then poisons the sum ([02.8 §2.3](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb)). `torch.where` *selects* — it never does arithmetic on the masked value — so it's `NaN`-safe:

In [ ]:
y = torch.tensor([[1., 3., 5.], [2., 4., 6.]])       # decoder output (batch=2, chan=3)
t = torch.zeros(2, 3); t[1, 1] = float("nan")        # one removed-channel NaN in target
mask = ~torch.isnan(t)

naive = (mask.float() * (y - t)).pow(2).sum()        # mask * diff  → NaN·0 = NaN
safe  = torch.where(mask, y - t, torch.zeros_like(y)).pow(2).sum()   # select → clean
print("mask * diff  (naive):", naive.item(), " ← poisoned")
print("torch.where  (safe): ", safe.item(), " ← correct masked sum-of-squares")

### 2.4 — The normalization the plan got *wrong* (verify empirically!)

Here's the flagship parity lesson of this whole curriculum. The masked sum-of-squares has to be normalized — but *by what*? Two candidates:

* divide by **`mask.sum()`** (the number of *unmasked* elements) — what the migration plan's Critical Note #38 originally said, and what "mean squared error" intuitively suggests;
* divide by **`batch_size`** (the observation-axis length) — what MATLAB's `l2loss` *actually* does, regardless of the mask.

They differ, and the difference silently rescales the reconstruction gradient by a data-dependent factor. The only way to know which is right is to *probe MATLAB directly* — not to trust the plan. The probe:

In [ ]:
from neural_data_decoding.training.losses.elbo import masked_mse_reconstruction_loss

loss = masked_mse_reconstruction_loss(y, t, batch_dim=0)
sq = safe.item()                                     # masked sum-of-squares = 75
print(f"masked sum-of-squares: {sq}   batch_size: {y.shape[0]}   mask.sum(): {int(mask.sum())}")
print(f"  divide by batch_size (CORRECT): 0.5 * {sq} / {y.shape[0]} = {0.5*sq/y.shape[0]}")
print(f"  divide by mask.sum() (the plan, WRONG): 0.5 * {sq} / {int(mask.sum())} = {0.5*sq/int(mask.sum())}")
print(f"actual loss = {loss.item()}  → confirms batch_size normalization.")

`18.75`, not `7.5`. MATLAB's `l2loss(Y, T, Mask=~isnan(T))` divides by the batch size (2), not the unmasked count (5) — the probe `l2loss == 37.5 == 75/2` settled it. Had the port followed the plan and divided by `mask.sum()`, the reconstruction gradient would have been scaled by `mask.sum()/batch_size` — a factor that *changes with how many channels each trial happens to have removed*. Training would have "worked" (no crash, plausible curves) while silently diverging from MATLAB. This is exactly why the project's discipline is **verify parity empirically against fixtures; don't trust the plan's numbers** — the empirical probe is the source of truth, and here it overturned the written plan.

### 2.5 — Per-channel reconstruction is telemetry only

The loss also computes a *per-channel* reconstruction (how well each area reconstructs) — but that's **detached** (Critical Note #33): logged for monitoring, never backpropagated ([02.5 §2.4](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb)). Only the single batch-normalized scalar drives the gradient. Backpropagating the per-channel losses too would double-count reconstruction in the objective.

### 2.6 — The two-tensor contract, drawn

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data = np.array([[1.0, np.nan, 3.0, 4.0],
                 [np.nan, 5.0, 6.0, np.nan],
                 [7.0, 8.0, np.nan, 9.0]])

fig, (axIn, axLoss) = plt.subplots(1, 2, figsize=(11, 4))

# Left: input path — NaN → 0 (encoder-safe)
inp = np.where(np.isnan(data), 0.0, data)
axIn.imshow(inp, cmap="Blues", vmin=0, vmax=9)
for (r, c), v in np.ndenumerate(data):
    txt = "0\n(was NaN)" if np.isnan(v) else f"{v:.0f}"
    axIn.text(c, r, txt, ha="center", va="center",
              color="crimson" if np.isnan(v) else "black", fontsize=9, fontweight="bold")
axIn.set_title("(a) INPUT to encoder\nNaN → 0 (must be finite)"); axIn.set_xticks([]); axIn.set_yticks([])

# Right: loss path — mask (which cells are scored)
maskimg = (~np.isnan(data)).astype(float)
axLoss.imshow(maskimg, cmap="Greens", vmin=0, vmax=1)
for (r, c), v in np.ndenumerate(data):
    txt = "skip" if np.isnan(v) else "score"
    axLoss.text(c, r, txt, ha="center", va="center",
                color="gray" if np.isnan(v) else "darkgreen", fontsize=9, fontweight="bold")
axLoss.set_title("(b) LOSS mask = ~isnan(target)\nNaN positions skipped"); axLoss.set_xticks([]); axLoss.set_yticks([])
plt.tight_layout(); plt.show()
print("SAME NaN positions, TWO roles: zeroed for the encoder, skipped by the loss.")

The figure is the whole idea: the *identical* NaN positions do double duty — zeroed out so the encoder can run (left), and skipped so the loss doesn't penalize them (right). One set of holes, two tensors, two treatments.

## Section 3 — The `neural_data_decoding` implementation

`masked_mse_reconstruction_loss` — the mask, the `torch.where`, the batch-size normalization:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/losses/elbo.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "mask = ~torch.isnan(y_target)" in line)
for k in range(i, i + 7):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `mask = ~torch.isnan(y_target)` — the mask comes from the **target's** NaNs (the NaN-preserving tensor, §2.2), not the input.
* `torch.where(mask, y_pred - y_target, 0)` — the `NaN`-safe select (§2.3), with an inline comment stating exactly why (`NaN * 0 == NaN`).
* `0.5 * sq_sum / batch_size` — the empirically-verified normalization (§2.4). The module docstring shows the MATLAB probe (`l2loss == 37.5 == 75/2`) that overturned the plan.
* The `if y_pred.shape != y_target.shape` guard — the two tensors must align position-for-position, or the mask maps to the wrong cells.
* `NaNToZero` (`models/layers/nan_to_zero.py`) is layer (a); this function is layer (b). The [composite forward](../04_architecture/04.1_architecture_string_dispatcher.ipynb) applies `NaNToZero` as its leading transform, then hands the NaN-preserving target to this loss — the two-tensor contract wired end-to-end.

## Section 4 — Hands-on exercises

### Exercise 4.1 — drop layer (a), watch the poison spread

Feed a NaN-containing input straight through a `Linear` (skipping `NaNToZero`) and show the *entire* output is NaN — one hole poisons everything.

In [ ]:
# Reveal:
import torch.nn as nn
lin = nn.Linear(3, 4)
x_bad = torch.tensor([[1.0, float("nan"), 3.0]])     # one NaN channel
print("output WITHOUT NaNToZero:", lin(x_bad).tolist(), "← all NaN")
print("output WITH NaNToZero:   ", lin(NaNToZero()(x_bad)).tolist(), "← finite")

### Exercise 4.2 — the normalization changes the gradient

Compute the reconstruction loss's gradient w.r.t. `y_pred` under batch-size norm vs mask.sum() norm, and show they differ by the factor `mask.sum()/batch_size`.

In [ ]:
# Reveal:
def recon_grad(norm):
    yp = y.clone().requires_grad_(True)
    sq = torch.where(mask, yp - t, torch.zeros_like(yp)).pow(2).sum()
    (0.5 * sq / norm).backward()
    return yp.grad.abs().sum().item()

g_batch = recon_grad(y.shape[0])          # batch_size = 2 (correct)
g_masksum = recon_grad(int(mask.sum()))   # mask.sum() = 5 (the plan)
print(f"grad magnitude, batch-norm:   {g_batch:.4f}")
print(f"grad magnitude, mask.sum-norm: {g_masksum:.4f}")
print(f"ratio: {g_batch / g_masksum:.3f}  ==  mask.sum()/batch = {int(mask.sum())/y.shape[0]:.3f}")
print("→ the wrong normalization scales EVERY reconstruction gradient by this factor.")

## Section 5 — Common errors

### The whole forward pass is NaN

Layer (a) is missing — a `NaN` reached the network ([02.8 §2.2](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb)). Apply `NaNToZero` before the encoder. The composite does this by default; a hand-built forward might skip it.

### The loss is NaN even though the input was zeroed

Layer (b) used `mask * diff` instead of `torch.where` (§2.3), or the target was NaN-zeroed too (then the mask is all-True and removed channels get penalized). The loss needs the *NaN-preserving* target.

### Reconstruction gradient magnitude is off vs MATLAB

The normalization (§2.4). Divide by `batch_size`, not `mask.sum()`. This is the one the plan itself got wrong — trust the empirical probe.

### Both tensors NaN-zeroed (or both NaN-preserving)

The contract is *asymmetric*: input zeroed, target preserved. Zero both and the loss can't find the removed channels; preserve both and the encoder gets a NaN. Two tensors, two treatments (§2.6).

### Per-channel reconstruction affecting training

It's detached telemetry (§2.5, Critical Note #33). If per-channel terms are in the backward graph, reconstruction is double-counted. Only the batch-normalized scalar backpropagates.

## Section 6 — Further reading

- [`src/neural_data_decoding/training/losses/elbo.py`](../../src/neural_data_decoding/training/losses/elbo.py) — `masked_mse_reconstruction_loss` + the Critical Note #38 probe in the module docstring.
- [`src/neural_data_decoding/models/layers/nan_to_zero.py`](../../src/neural_data_decoding/models/layers/nan_to_zero.py) — layer (a).
- [02.8 NaN handling](../02_numpy_and_pytorch_basics/02.8_nan_handling.ipynb) — the IEEE `NaN` semantics this two-layer strategy is built around.

Next up: **[06.11 single total loss → three subnetworks](06.11_single_total_loss_three_subnetworks.ipynb)** — how one scalar's gradient reaches the encoder, decoder, and classifier.